In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/handwritten-equation-solver-data'

In [ ]:
!git config --global user.email "rahulkumar0909007@gmail.com"
!git config --global user.name "Rahul"

In [ ]:
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')

!git clone https://{TOKEN}@github.com/Rahullogic/handwritten-equation-solver.git
%cd handwritten-equation-solver

In [ ]:
!pwd

In [ ]:
%cd /content/handwritten-equation-solver
!pwd   # should print /content/handwritten-equation-solver

In [ ]:
%%writefile .gitignore
data/
checkpoints/
*.pt
*.pth
__pycache__/
.ipynb_checkpoints/
*.pyc
.DS_Store

In [ ]:
!cat .gitignore

In [ ]:
!git add .gitignore
!git commit -m "Add .gitignore"
!git push

In [ ]:
!mkdir -p checkpoints
!touch checkpoints/dummy_test.pt
!git status

In [ ]:
!rm checkpoints/dummy_test.pt

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [ ]:
!kaggle datasets download -d xainano/handwrittenmathsymbols

In [ ]:
!ls /content/handwritten-equation-solver

In [ ]:
import shutil
shutil.move('/content/handwritten-equation-solver/handwrittenmathsymbols.zip', f'{DATA_DIR}/handwrittenmathsymbols.zip')

In [ ]:
DATA_DIR = '/content/drive/MyDrive/handwritten-equation-solver-data'

In [ ]:
import zipfile
with zipfile.ZipFile(f'{DATA_DIR}/handwrittenmathsymbols.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/symbols_data')

In [ ]:
!apt-get install unrar -y

In [ ]:
!unrar x /content/symbols_data/data.rar /content/symbols_data/full_extracted/

In [ ]:
!ls {DATA_DIR}

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
torch.manual_seed(42)

In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28,28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5),(0.5))
])

In [ ]:
data_dir = '/content/symbols_data/full_extracted/extracted_images'

In [ ]:
dataset= datasets.ImageFolder(root= data_dir, transform= transform)

In [ ]:
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


print(f"Successfully loaded {len(dataset)} images.")
print(f"The model will learn to recognize these classes: {dataset.classes}")

In [ ]:
len(dataset.classes)

In [ ]:
class MyNN(nn.Module):

  def __init__(self, input_features, num_classes):

    super().__init__()

    self.features= nn.Sequential(
        nn.Conv2d(input_features, 32, kernel_size=3, padding= 'same'),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(32, 64, kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

    self.classifier= nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*7*7, 128),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(64,num_classes)
    )

  def forward(self, X):

    X= self.features(X)
    X= self.classifier(X)

    return X

In [ ]:
learning_rate=0.01
epochs= 10

In [ ]:
model= MyNN(1,len(dataset.classes))
model.to(device)

criterion= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [ ]:
for epoch in range(epochs):

  model.train()
  running_loss = 0.0
  correct_predictions = 0
  total_samples = 0

  for images,labels in dataloader:

    images= images.to(device)
    labels= labels.to(device)

    outputs= model(images)

    loss= criterion(outputs, labels)

    optimizer.zero_grad()
    loss.backward()

    optimizer.step()

    running_loss += loss.item() * images.size(0)

    _, predicted = torch.max(outputs, 1)
    total_samples += labels.size(0)
    correct_predictions += (predicted == labels).sum().item()

  epoch_loss = running_loss / total_samples
  epoch_acc = (correct_predictions / total_samples) * 100

  print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

print("Training Completed");





In [ ]:
model.eval()

In [ ]:

save_path = '/content/drive/MyDrive/Handwritten_equation_solver/symbol_classifier.pth'

torch.save(model.state_dict(), save_path)
print("Model saved successfully!")

In [ ]:
#Now splitting the data into training data and validation data. Also adding one convolutional layer to model and increasing epochs

In [ ]:
from torch.utils.data import random_split

train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_dataloader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_dataloader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
class MyNN(nn.Module):

  def __init__(self, input_features, num_classes):

    super().__init__()

    self.features= nn.Sequential(
        nn.Conv2d(input_features, 32, kernel_size=3, padding= 'same'),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(32, 64, kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(64, 128, kernel_size= 3, padding= 'same'),
        nn.ReLU(),
        nn.BatchNorm2d(128),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

    self.classifier= nn.Sequential(
        nn.Flatten(),
        nn.Linear(128*3*3, 128),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(64,num_classes)
    )

  def forward(self, X):

    X= self.features(X)
    X= self.classifier(X)

    return X

In [ ]:
model= MyNN(1,len(dataset.classes))
model.to(device)

criterion= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [ ]:
epochs2=25

In [ ]:
for epoch in range(epochs2):

  model.train()
  running_loss = 0.0
  correct_predictions = 0
  total_samples = 0

  for images,labels in train_dataloader:

    images= images.to(device)
    labels= labels.to(device)

    outputs= model(images)

    loss= criterion(outputs, labels)

    optimizer.zero_grad()
    loss.backward()

    optimizer.step()

    running_loss += loss.item() * images.size(0)

    _, predicted = torch.max(outputs, 1)
    total_samples += labels.size(0)
    correct_predictions += (predicted == labels).sum().item()

  epoch_loss = running_loss / total_samples
  epoch_acc = (correct_predictions / total_samples) * 100

  print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

  model.eval()
  val_correct = 0
  val_total = 0
  with torch.no_grad():
      for images, labels in val_dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
  val_acc = (val_correct / val_total) * 100

  print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("Training Completed");





In [ ]:
save_path = '/content/drive/MyDrive/Handwritten_equation_solver/symbol_classifier_82class.pth'
torch.save(model.state_dict(), save_path)
print("Model saved successfully!")

In [ ]:
!pip install nbstripout
!nbstripout Handwritten_equation_solver.ipynb

In [ ]:
from google.colab import _message
import json

live_notebook = _message.blocking_request('get_ipynb', timeout_sec=30)['ipynb']

with open('/content/handwritten-equation-solver/Handwritten_equation_solver.ipynb', 'w') as f:
    json.dump(live_notebook, f)

In [ ]:
!git add Handwritten_equation_solver.ipynb
!git commit -m "Retrained on full 82-class dataset, added validation split"
!git push